In [1]:
# ADK's Built-in Toolbox: The Foundation
# Let’s walk through an example where a user asks, “What food festivals are happening in London next month?” 
# The agent uses the `google_search` tool to fetch the latest events, then updates the itinerary and packing list accordingly. 
# This isn’t just static knowledge—it’s live, dynamic, and tailored to the user’s needs.

In [2]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# (optional) Verify installation
%pip show google-adk

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [5]:
# Import Required Modules
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types
from IPython.display import Markdown, display

In [6]:

# Latest Events Agent: Definition
# The instruction set is the heart of the agent. Make it specific, structured, and safe.
latest_events_agent = Agent(
    name="latest_events_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that gives the latest events for a destination.",
    instruction="""
        You are a travel assistant. 
        When a user asks about events, use the google_search tool to find current or upcoming events and summarize them in a user-friendly way.
""",
    tools=[google_search],
)
# NOTE: In future lessons, you will integrate external tools (e.g., weather APIs) for even more personalized lists.

In [7]:
# Latest Events Agent: Session Setup and Run
# Define the app name, user ID, session ID, and user input for the packing list.
APP_NAME = "wanderwise_app"
USER_ID = "user_001"
SESSION_ID = "wanderwise_session_001"
USER_INPUT = " “What food festivals are happening in London next month?”"

# Create a session service to manage user sessions.
session_service = InMemorySessionService()

# Create a session (async).
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Create a content object that contains the user's request.
content = types.Content(role="user", parts=[types.Part(text=USER_INPUT)])

# Create the runner for your packing list agent.
runner = Runner(
    agent=latest_events_agent,
    app_name=APP_NAME,
    session_service=session_service
)

# Invoke the agent with the user's input and print the response.
events = runner.run(
    user_id=USER_ID,
    session_id=SESSION_ID,
    new_message=content
)

for event in events:
    if event.is_final_response():
        response_text = event.content.parts[0].text
        display(Markdown(f"{response_text}"))

Here are some food festivals happening in London next month, June 2025:

*   **Taste of London:** From June 18 - June 22, 2025, Regent's Park will host the Taste of London Food Festival. This festival will showcase top-tier restaurants and a variety of drinks.
*   **Indian Street Food Festival - London - Summer 2025:** On June 22, 2025, OMNOM Restaurant will host the Indian Street Food Festival. This festival will feature a wide variety of delicious dishes from different regions of India, live music, and cultural experiences.

I hope you have a great time at the food festivals!


In [8]:
# 🎉 Congratulations!
# You’ve just built and run an agent that uses the ADK built-in google_search tool to fetch live event data for your itinerary. 
# Try modifying the prompt or agent instructions, and see how the results change!